### Set Up

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random


In [3]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


### Load dataset

In [4]:
# Downloading an abbreviated collection of Shakespeare’s work
filename = keras.utils.get_file(origin=("https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"),)
shakespeare = open(filename, "r").read()
print(shakespeare[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



### Data preprocess

In [5]:
# Splitting text into chunks for language model training
sequence_length = 100
def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]
features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [6]:
features[:1], features[-1:]

(['First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'],
 ["\nNoble Sebastian,\nThou let'st thy fortune sleep--die, rather; wink'st\nWhiles thou art waking."])

In [7]:
labels[:1], labels[-1:]

(['irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '],
 ["Noble Sebastian,\nThou let'st thy fortune sleep--die, rather; wink'st\nWhiles thou art waking.\n"])

In [8]:
x, y = next(dataset.as_numpy_iterator())
x, y

(b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou',
 b'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou ')

In [9]:
# Learning a character-level vocabulary with the TextVectorization layer
tokenizer = keras.layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

In [10]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

In [11]:
dataset = dataset.map(lambda features, labels: (tokenizer(features), tokenizer(labels)), num_parallel_calls=8,)
training_data = dataset.shuffle(10_000).batch(64).cache()

### Model

In [12]:
# Building a miniature language model
embedding_dim = 256
hidden_dim = 1024
inputs = keras.layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = keras.layers.GRU(hidden_dim, return_sequences=True)(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 256)       │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 100, 1024)      │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100, 67)        │        68,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

### Training

In [13]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="loss",
    restore_best_weights=True,
    patience=3,
)

In [14]:
# Training a miniature language mode
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=100, callbacks=[early_stopping])

Epoch 1/200


C:\Users\PRASHANTH N\PycharmProjects\MTechSem2\gpu\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:853: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n = torch._VF.gru(


175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 2.7459 - sparse_categorical_accuracy: 0.2769
Epoch 2/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - loss: 2.0403 - sparse_categorical_accuracy: 0.4041
Epoch 3/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - loss: 1.7946 - sparse_categorical_accuracy: 0.4692
Epoch 4/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - loss: 1.6502 - sparse_categorical_accuracy: 0.5069
Epoch 5/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - loss: 1.5571 - sparse_categorical_accuracy: 0.5318
Epoch 6/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - loss: 1.4911 - sparse_categorical_accuracy: 0.5489
Epoch 7/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - loss: 1.4409 - sparse_categorical_accuracy: 0.5615
Epoch 8/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - loss: 1.3982 - sparse_categorical_accuracy: 0.5728
Epoch 9/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - loss: 1.3611 - sparse_categorical_accuracy: 0.5821
Epoch 10/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms

### Inference

In [15]:
# Modifying the language model for autoregressive inference
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = keras.layers.GRU(hidden_dim, return_state=True)(x, initial_state=input_state)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(inputs=(inputs, input_state), outputs=(outputs, output_state), )
generation_model.set_weights(model.get_weights())
generation_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 256)    │     17,152 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state (InputLayer)  │ (None, 1024)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, 1024),    │  3,938,304 │ embedding_1[0][0… │
│                     │ (None, 1024)]     │            │ state[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 67)        │     68,675 │ gru_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))
prompt = """Hi"""

In [17]:
# Using a fixed prompt to compute a language model’s starting state
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)
    print(predictions, state)

[[4.1182737e-09 9.9895416e-09 4.7092787e-03 5.1559854e-01 1.2321683e-04
  2.0871010e-02 1.5572391e-01 2.1662123e-05 2.2855424e-04 2.8074392e-05
  1.0560891e-05 2.3602517e-02 2.3046767e-03 1.4119423e-05 9.3054859e-06
  1.4813924e-03 1.2822644e-04 2.7191153e-04 2.9840893e-03 2.6795084e-05
  4.9952955e-06 1.6971610e-05 2.7710646e-06 1.8796505e-03 1.7000217e-05
  1.5210300e-06 3.4718219e-02 4.3122042e-03 8.5352376e-02 6.7942779e-07
  6.4958334e-07 6.4390892e-04 6.9219171e-04 7.5297281e-02 5.3964917e-02
  4.0123708e-04 1.2043319e-03 5.7087007e-05 5.2763680e-05 1.2108751e-03
  1.5510422e-03 3.4963875e-04 5.8601857e-03 3.7145735e-05 9.3828223e-04
  1.0752622e-03 2.1210700e-04 8.5727661e-05 4.0969197e-04 5.1109157e-05
  2.4571907e-04 4.4267472e-06 3.1376391e-04 7.8677840e-04 1.7891827e-05
  3.2660140e-05 6.5998611e-06 1.5814347e-05 4.5005316e-07 2.0553516e-07
  1.4313496e-06 1.2558172e-05 1.8166373e-05 1.2384202e-06 2.4811679e-06
  1.7520826e-08 2.0434879e-07]] [[-0.41794857 -0.17804323  0.352

In [18]:
# Predicting with the language model a token at a time
generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [19]:
generated_ids

[8,
 2,
 29,
 11,
 21,
 3,
 8,
 18,
 2,
 17,
 5,
 15,
 2,
 19,
 5,
 15,
 13,
 14,
 2,
 25,
 6,
 17,
 2,
 16,
 17,
 2,
 20,
 6,
 4,
 7,
 3,
 9,
 27,
 12,
 12,
 54,
 23,
 35,
 47,
 2,
 33,
 49,
 41,
 28,
 36,
 49,
 2,
 23,
 55,
 26,
 12,
 31,
 7,
 6,
 10,
 30,
 8,
 18,
 2,
 22,
 5,
 5,
 14,
 2,
 20,
 9,
 11,
 3,
 10,
 14,
 18,
 2,
 6,
 10,
 14,
 2,
 14,
 3,
 8,
 25,
 3,
 9,
 6,
 4,
 3,
 2,
 4,
 5,
 9,
 16,
 3,
 10,
 4,
 2,
 23,
 2,
 16,
 11,
 8,
 4,
 6,
 30,
 3,
 27,
 12,
 12,
 54,
 23,
 35,
 47,
 2,
 36,
 23,
 39,
 43,
 28,
 36,
 49,
 2,
 23,
 23,
 23,
 26,
 12,
 41,
 7,
 6,
 4,
 2,
 11,
 8,
 2,
 4,
 7,
 3,
 2,
 24,
 6,
 8,
 4,
 6,
 9,
 14,
 46,
 2,
 10,
 3,
 29,
 3,
 9,
 2,
 4,
 5,
 2,
 9,
 3,
 16,
 3,
 16,
 24,
 3,
 9,
 2,
 16,
 5,
 9,
 3,
 18,
 12,
 28,
 10,
 14,
 2,
 6,
 8,
 2,
 16,
 17,
 2,
 25,
 9,
 5,
 25,
 3,
 9,
 2,
 11,
 8,
 2,
 4,
 7,
 3,
 2,
 9,
 11,
 22,
 7,
 4,
 2,
 55,
 11,
 10,
 21,
 3,
 10,
 4,
 11,
 5,
 27,
 12,
 12,
 45,
 23,
 34,
 35,
 49,
 33,
 38,
 38,
 34,
 26,
 1

In [20]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

His vices, you would pay my father.

KING EDWARD IV:
Thanks, good friend, and desperate torment I mistake.

KING RICHARD III:
What is the bastard? never to remember more,
And as my proper is the right Vincentio.

BIONDELLO:
The mother were a book in a 
